## Introduction to Binary Search Trees (BSTs)

A **Binary Search Tree** is a fundamental data structure that combines the benefits of a sorted array with efficient insertion and deletion operations. Each node in a BST has at most two children: a **left child** and a **right child**.

### Key Properties of BSTs:
1. **Left Subtree Property**: All nodes in the left subtree have values **less than** the parent node
2. **Right Subtree Property**: All nodes in the right subtree have values **greater than** the parent node
3. **No Duplicates**: Each value appears only once (typically)
4. **Balanced vs Unbalanced**: A balanced tree has O(log n) operations, while unbalanced can degrade to O(n)

### Common Operations:
- **Search**: Find a value by traversing left or right based on comparisons
- **Insertion**: Add a new node maintaining BST properties
- **Deletion**: Remove a node without breaking BST ordering
- **Traversal**: Visit all nodes in specific orders (in-order, pre-order, post-order)

### Why BSTs Matter:
- **Efficient Search**: Average O(log n) time complexity
- **Practical Applications**: Databases, file systems, auto-complete features
- **Foundation**: Understanding BSTs leads to learning advanced structures like AVL trees and Red-Black trees


## How to Play — Interactive BST Explorer

### Controls
| Key | Action |
|-----|--------|
| **W** | Move up |
| **A** | Move left |
| **S** | Move down |
| **D** | Move right |
| **E** | Interact with a nearby node or the wizard |

### Scene Layout
The forest level mirrors a real **7-node balanced BST**:

<pre>
          [ 50 ]           ← Root (blue square)
         /       \
      [25]       [75]      ← Internal nodes (blue circles)
      /  \       /  \
   [12] [37]  [62] [87]   ← Leaf nodes (blue circles)

        🧙 BST Mentor      ← AI wizard at the bottom
</pre>

Walk to any node and press **E** to cycle through its dialogue.  
Walk to the **BST Mentor** wizard at the bottom and press **E** to open the AI chat — type any BST question!

### Learning Objectives
1. **Root Node (50)** — understand why every search starts here
2. **Internal Nodes (25, 75)** — see how each subtree independently obeys BST ordering
3. **Leaf Nodes (12, 37, 62, 87)** — learn insertion points and the simplest deletion case
4. **BST Mentor** — ask free-form AI questions about traversal, deletion, balancing, and Big-O

### SRP Design in This Game
This level follows the **Single Responsibility Principle**:
- `BSTLevel` — only configures which objects exist and where
- `NPC` — only handles collision detection and dialogue cycling
- `AiNpc` — only manages the AI chatbot UI and backend calls
- `Player` — only handles movement and animation
- `GameEnvBackground` — only draws the background image
- `GameControl` — only orchestrates the main game loop


In [ ]:
%%js

// GAME_RUNNER: Explore a Binary Search Tree! Use WASD to walk through the forest and press E near each node to learn BST concepts | hide_edit: true

import GameControl from '/assets/js/GameEnginev1.1/essentials/GameControl.js';
import GameEnvBackground from '/assets/js/GameEnginev1.1/essentials/GameEnvBackground.js';
import Player from '/assets/js/GameEnginev1.1/essentials/Player.js';
import NPC from '/assets/js/GameEnginev1.1/essentials/Npc.js';
import AiNpc from '/assets/js/GameEnginev1.1/essentials/AiNpc.js';

class BSTLevel {
  constructor(gameEnv) {
    const path   = gameEnv.path;
    const width  = gameEnv.innerWidth;
    const height = gameEnv.innerHeight;

    // only supplies the background image path
    const bgData = {
      name: 'bst_forest_bg',
      src: path + '/images/gamify/forest.png',
    };

    // ── Player (only supplies spawn/animation config for the hero) ────
    const playerData = {
      id: 'Hero',
      src: path + '/images/gamify/chillguy.png',
      SCALE_FACTOR: 5,
      STEP_FACTOR: 1000,
      ANIMATION_RATE: 50,
      INIT_POSITION: { x: width * 0.18, y: height * 0.80 },
      pixels:    { height: 512, width: 384 },
      orientation: { rows: 4, columns: 3 },
      down:      { row: 0, start: 0, columns: 3 },
      downRight: { row: 1, start: 0, columns: 3, rotate:  Math.PI / 16 },
      downLeft:  { row: 2, start: 0, columns: 3, rotate: -Math.PI / 16 },
      right:     { row: 1, start: 0, columns: 3 },
      left:      { row: 2, start: 0, columns: 3 },
      up:        { row: 3, start: 0, columns: 3 },
      upRight:   { row: 1, start: 0, columns: 3, rotate: -Math.PI / 16 },
      upLeft:    { row: 2, start: 0, columns: 3, rotate:  Math.PI / 16 },
      hitbox: { widthPercentage: 0.45, heightPercentage: 0.2 },
      keypress: { up: 87, left: 65, down: 83, right: 68 },
    };

    // ═══════════════════════════════════════════════════════════════════════
    // BST Node Layout — visual map of the 7-node balanced tree:
    //
    //              [ 50 ] ← Root  (bluesquare)      y ≈ 8 %
    //             /       \
    //          [25]       [75]   ← Internal nodes   y ≈ 32 %
    //          /  \       /  \
    //        [12] [37]  [62] [87]  ← Leaves         y ≈ 58 %
    //
    //  Wizard AI mentor sits near the bottom:                y ≈ 78 %
    // ═══════════════════════════════════════════════════════════════════════

    // Root node uses bluesquare (SCALE_FACTOR 5); circle nodes use SCALE_FACTOR 6
    // so circles render slightly smaller than the square to match visual proportions.
    // All node hitboxes use 0.8 so the interaction zone covers the full visible sprite.
    const rootNode = {
      id: 'Root Node (50)',
      src: path + '/images/gamify/bluesquare.png',
      SCALE_FACTOR: 5,
      INIT_POSITION: { x: width * 0.46, y: height * 0.05 },
      orientation: { rows: 1, columns: 1 },
      down: { row: 0, start: 0, columns: 1 },
      hitbox: { widthpercentage: 0.4, heightpercentage: 0.4 },
      greeting: 'I am the entry point of the entire BST. All searches begin here!',
      dialogues: [
        'Every search, insertion, and deletion starts with me!',
        'Values less than 50 travel left; values greater than 50 travel right.',
        'I have exactly two children: 25 (left) and 75 (right).',
        'A balanced BST like this gives O(log n) time for search — very fast!',
        'If you remove me, my in-order successor or predecessor takes over.',
      ],
      reaction: function() { if (this.dialogueSystem) this.showReactionDialogue(); },
      interact: function() { if (this.dialogueSystem) this.showRandomDialogue(); },
    };

    const leftChild = {
      id: 'Left Child (25)',
      src: path + '/images/gamify/bluecircle.png',
      SCALE_FACTOR: 6,
      INIT_POSITION: { x: width * 0.22, y: height * 0.30 },
      orientation: { rows: 1, columns: 1 },
      down: { row: 0, start: 0, columns: 1 },
      hitbox: { widthpercentage: 0.4, heightpercentage: 0.4 },
      greeting: 'I live left of the Root because 25 < 50.',
      dialogues: [
        'I am less than Root, so I belong in the left subtree.',
        'My own children are 12 (left) and 37 (right).',
        'BST order: 12 < 25 < 37 — my subtree is perfectly sorted too!',
        'In-order traversal visits me between my children: 12 → 25 → 37.',
        'To search for 30: go left at Root, then go right — reach 37 and not found.',
      ],
      reaction: function() { if (this.dialogueSystem) this.showReactionDialogue(); },
      interact: function() { if (this.dialogueSystem) this.showRandomDialogue(); },
    };

    const rightChild = {
      id: 'Right Child (75)',
      src: path + '/images/gamify/bluecircle.png',
      SCALE_FACTOR: 6,
      INIT_POSITION: { x: width * 0.68, y: height * 0.30 },
      orientation: { rows: 1, columns: 1 },
      down: { row: 0, start: 0, columns: 1 },
      hitbox: { widthpercentage: 0.4, heightpercentage: 0.4 },
      greeting: 'I live right of the Root because 75 > 50.',
      dialogues: [
        'I am greater than Root, so I belong in the right subtree.',
        'My children are 62 (left) and 87 (right).',
        'To insert 80: Root →right→ me →right→ 87 →right→ new node 80.',
        'In-order traversal visits my subtree: 62 → 75 → 87.',
        'Deleting me, which has two children, requires finding my in-order successor: 87.',
      ],
      reaction: function() { if (this.dialogueSystem) this.showReactionDialogue(); },
      interact: function() { if (this.dialogueSystem) this.showRandomDialogue(); },
    };

    const leafLL = {
      id: 'Leaf Node (12)',
      src: path + '/images/gamify/bluecircle.png',
      SCALE_FACTOR: 6,
      INIT_POSITION: { x: width * 0.08, y: height * 0.56 },
      orientation: { rows: 1, columns: 1 },
      down: { row: 0, start: 0, columns: 1 },
      hitbox: { widthpercentage: 0.4, heightpercentage: 0.4 },
      greeting: 'I have no children — I am the smallest value in this tree!',
      dialogues: [
        'I am a leaf — no children, no subtree.',
        'I am the minimum of this entire BST.',
        'Path from root to me: 50 → left → 25 → left → 12.',
        'Deleting a leaf is the easiest BST operation — just remove the node!',
        'My depth is 2 (distance from root). This balanced tree has max depth 2.',
      ],
      reaction: function() { if (this.dialogueSystem) this.showReactionDialogue(); },
      interact: function() { if (this.dialogueSystem) this.showRandomDialogue(); },
    };

    const leafLR = {
      id: 'Leaf Node (37)',
      src: path + '/images/gamify/bluecircle.png',
      SCALE_FACTOR: 6,
      INIT_POSITION: { x: width * 0.34, y: height * 0.56 },
      orientation: { rows: 1, columns: 1 },
      down: { row: 0, start: 0, columns: 1 },
      hitbox: { widthpercentage: 0.4, heightpercentage: 0.4 },
      greeting: 'I am between 25 and 50, so I am the right child of 25.',
      dialogues: [
        'I am greater than my parent but less than Root.',
        'Path from root: 50 → left → right.',
        'In-order traversal order: 12, 25, 37, 50, 62, 75, 87 — that is sorted!',
        'I am the in-order predecessor of Root.',
        'If Root were deleted, I could replace it since I am its predecessor.',
      ],
      reaction: function() { if (this.dialogueSystem) this.showReactionDialogue(); },
      interact: function() { if (this.dialogueSystem) this.showRandomDialogue(); },
    };

    const leafRL = {
      id: 'Leaf Node (62)',
      src: path + '/images/gamify/bluecircle.png',
      SCALE_FACTOR: 6,
      INIT_POSITION: { x: width * 0.56, y: height * 0.56 },
      orientation: { rows: 1, columns: 1 },
      down: { row: 0, start: 0, columns: 1 },
      hitbox: { widthpercentage: 0.4, heightpercentage: 0.4 },
      greeting: 'I am between Root and my parent — right subtree, left branch.',
      dialogues: [
        'I am greater than Root but less than my parent.',
        'Path from root: 50 → right → left.',
        'BST search for 62: 62 > 50 go right, 62 < 75 go left — found in 2 comparisons!',
        'I am the in-order successor of Root.',
        'If Root is deleted, I would naturally replace it.',
      ],
      reaction: function() { if (this.dialogueSystem) this.showReactionDialogue(); },
      interact: function() { if (this.dialogueSystem) this.showRandomDialogue(); },
    };

    const leafRR = {
      id: 'Leaf Node (87)',
      src: path + '/images/gamify/bluecircle.png',
      SCALE_FACTOR: 6,
      INIT_POSITION: { x: width * 0.78, y: height * 0.56 },
      orientation: { rows: 1, columns: 1 },
      down: { row: 0, start: 0, columns: 1 },
      hitbox: { widthpercentage: 0.4, heightpercentage: 0.4 },
      greeting: 'I am the largest value in this tree — always go right to reach me!',
      dialogues: [
        'I am the maximum of this BST.',
        'Path from root: 50 → right → right. Found in 2 comparisons!',
        'Adding 90 here turns me into a parent; 90 becomes my right child.',
        'A BST of only right children is "skewed" — search degrades to O(n).',
        'AVL and Red-Black trees prevent skewing through automatic rebalancing.',
      ],
      reaction: function() { if (this.dialogueSystem) this.showReactionDialogue(); },
      interact: function() { if (this.dialogueSystem) this.showRandomDialogue(); },
    };

    const wizardData = {
      id: 'BST Mentor',
      greeting: "Hello, traveler! I am the BST Mentor. Ask me anything about Binary Search Trees!",
      src: path + '/images/gamify/wizard.png',
      SCALE_FACTOR: 7,
      ANIMATION_RATE: 10,
      pixels:    { height: 185, width: 163 },
      INIT_POSITION: { x: width * 0.46, y: height * 0.72 },
      orientation: { rows: 1, columns: 1 },
      down: { row: 0, start: 0, columns: 1 },
      hitbox: { widthPercentage: 0.2, heightPercentage: 0.2 },
      expertise: 'binary_search_trees',
      chatHistory: [],
      dialogues: [
        'Ask me how in-order traversal produces a sorted sequence!',
        'Try asking: "How do I delete a node with two children?"',
        'Curious about AVL trees vs regular BSTs? Just ask!',
        'Ask me about Big-O complexity for balanced vs skewed trees.',
        'Want to know how databases use BST-like structures for indexing?',
      ],
      knowledgeBase: {
        default: [
          {
            question: 'What is a Binary Search Tree?',
            answer: 'A Binary Search Tree, or BST, is a node-based tree where every left child holds a value smaller than its parent and every right child holds a value larger. This ordering enables O(log n) average-case search, insertion, and deletion.',
          },
          {
            question: 'What is in-order traversal?',
            answer: 'In-order traversal visits the left subtree, then the root, then the right subtree. For any valid BST this always yields all values in ascending sorted order.',
          },
          {
            question: 'How do you delete a node with two children?',
            answer: 'Find the in-order successor (smallest node in the right subtree) or predecessor (largest in the left subtree). Copy its value into the node you want to delete, then delete the successor/predecessor — which has at most one child, making it simple.',
          },
          {
            question: 'What is the time complexity of BST operations?',
            answer: 'Average case: O(log n) for search, insert, and delete on a balanced BST. Worst case: O(n) for a completely skewed tree (e.g., inserting already-sorted values). Balanced variants like AVL and Red-Black trees guarantee O(log n) worst case.',
          },
          {
            question: 'What is an AVL tree?',
            answer: 'An AVL tree is a self-balancing BST where the heights of the two child subtrees of any node differ by at most 1. After each insertion or deletion, rotations restore balance, guaranteeing O(log n) operations.',
          },
          {
            question: 'What is the difference between pre-order, in-order, and post-order traversal?',
            answer: 'Pre-order (Root → Left → Right) is useful for copying a tree. In-order (Left → Root → Right) yields sorted output. Post-order (Left → Right → Root) is used to safely delete a tree or evaluate expression trees.',
          },
        ],
      },
      reaction: function() {
        if (this.dialogueSystem) this.showReactionDialogue();
      },
      interact: function() {
        AiNpc.showInteraction(this);
      },
    };

    this.classes = [
      { class: GameEnvBackground, data: bgData    },
      { class: Player,            data: playerData },
      { class: NPC,               data: rootNode   },
      { class: NPC,               data: leftChild  },
      { class: NPC,               data: rightChild },
      { class: NPC,               data: leafLL     },
      { class: NPC,               data: leafLR     },
      { class: NPC,               data: leafRL     },
      { class: NPC,               data: leafRR     },
      { class: NPC,               data: wizardData },
    ];
  }
}

export const gameLevelClasses = [BSTLevel];
export { GameControl };
